<a href="https://colab.research.google.com/github/data-188-berkeley/hw3/blob/main/hw3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data 188 Homework 3

In this homework, you will learn to use PyTorch, a popular deep learning library used in both academia and industry.

Your "deliverables" for this homework will be the following:
- hw3.ipynb (this notebook!)
- hw3_gpu.ipynb
- student_data_cpu.pt
- student_data_gpu.pt
- model_q1_fashion_mnist.pth


## Clarifications Doc

In case we need to give any clarifications (or hints!) for the assignment, see this Google Doc: ["HW3 Clarifications"](https://docs.google.com/document/d/1kYPYE25w_gNUOeXCClbUpch35EysCLYmOQHyvU5ea50/edit?tab=t.0).
This doc will be regularly updated.

## General Tips

Unless otherwise instructed, don't modify cells aside from in between the blocks with `# BEGIN YOUR SOLUTION`, `# END YOUR SOLUTION`.

Modifying other code can result in failing autograder tests.

## Setup

Setup will be a little different for this assignment than previous assignments.
Namely, you will be doing all work within the Colab Jupyter notebooks, and submitting the notebooks (along with additional generated artifacts) as part of your Gradescope submission.

Suggested steps on setting things up:

1. **Download the assignment**. Go to the [hw3 github repository](https://github.com/data-188-berkeley/hw3), and download the repo code `hw3.zip` via the green "Code -> Download Zip" button to your local machine.
2. **Upload to Google Drive**. Extract the `hw3.zip` file, and upload the contents of the `hw3.zip` file to your Google Drive to: `My Drive/data188/hw3`.

At this point, your `My Drive/data188/hw3` should look like:

```
utils_public/
hw3.ipynb
hw3_gpu.ipynb
...
```

3. **Open notebook in Colab**. In Google Drive, navigate to `My Drive/data188/hw3`, and double-click the `hw3.ipynb` notebook to open it in Colab.

**Important:** At this point: please double check that you are working on the notebook located at `My Drive/data188/hw3/hw3.ipynb` before proceeding with the assignment!

In [ ]:
# Run this cell each time your kernel is disconnected or restarted
# Tip: it's always safe to re-run this, this will never delete data

import os
import sys

# basedir_course: All colab material for this course will live here.
#   Feel free to modify this if you'd like.
basedir_course = "/content/drive/MyDrive/data188"
# asn_name: name of assignment, eg hw0. This must match github repo name.
asn_name = "hw3"
rootdir_asn = os.path.join(basedir_course, asn_name)

# Fetch code to set up the assignment, then set the correct working directory
# (eg for python imports to work)
from google.colab import drive
drive.mount('/content/drive')

os.chdir(rootdir_asn)

# Validate that our current working directory is correct.
# This should output:
#   /content/drive/MyDrive/data188/hw3/
print("Current working directory: ", os.getcwd())
# Another check: let's double check that the files in hw0/ are in the current directory.
# This should output something like:
#   ['.git', 'LICENSE', 'hw3.ipynb', ...]
print("ls cwd: ", os.listdir(os.getcwd()))

# We also need to modify sys.path directly so that the current python session
# can access utils_public/
sys.path.insert(1, "./")

print("Setup done!")

In [ ]:
# ================================================================
# IMPORTANT: DO NOT MODIFY THIS CELL
# ================================================================

# First, some initial imports/helper functions
import math
import time
from collections import defaultdict

from typing import Any
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
from torch.utils.data import DataLoader

from utils_public.utils_autograder import save_student_data
from utils_public.utils_torch import get_model_device
from utils_public.utils_torch import count_model_parameters
from utils_public.utils_torch import check_metas_train_test
from utils_public.utils_torch import train_epochs
from utils_public.utils_torch import visualize_image_classifier_preds
from utils_public.utils_torch import visualize_image_classification_dataset
from utils_public.utils_torch import visualize_image_classification_dataloader

# ================================================================
# Code for managing student data (for autograder)
# ================================================================
OUTPATH_STUDENT_DATA_CPU = "student_data_cpu.pt"

# Initialize important data structures.
# submission dict, with keys:
#   ["output"][str name] -> data_to_check
student_data: dict[str, dict[str, Any]] = defaultdict(dict)

# Question 0: PyTorch Tutorials
(Recommended) First, run through the official [PyTorch Intro guide](https://docs.pytorch.org/tutorials/beginner/basics/intro.html).

You'll want to complete all sections, in order:
- ["Quickstart"](./pytorch_intro_notebooks/quickstart_tutorial.ipynb)
- ["Tensors"](./pytorch_intro_notebooks/tensorqs_tutorial.ipynb)
- ["Datasets & DataLoaders"](./pytorch_intro_notebooks/data_tutorial.ipynb)
- ["Transforms"](./pytorch_intro_notebooks/transforms_tutorial.ipynb)
- ["Build Model"](./pytorch_intro_notebooks/buildmodel_tutorial.ipynb)
- ["Autograd"](./pytorch_intro_notebooks/autogradqs_tutorial.ipynb)
- ["Optimization"](./pytorch_intro_notebooks/optimization_tutorial.ipynb)
- ["Save & Load Model"](./pytorch_intro_notebooks/saveloadrun_tutorial.ipynb)

Tip: Just in case, to avoid issues if/when the pytorch website changes, we have downloaded each section's notebooks, see: `hw3/pytorch_intro_notebooks/`.

# Question 1: Fashion MNIST Classifier [45 pts]
In this section, you will apply what you learned in this course (as well as the previous PyTorch tutorials) to build an end-to-end modular pipeline to train an image classification model on the Fashion-MNIST image classification dataset.

A pytorch ML pipeline consists of the following modular components:
- [torch.utils.data.Dataloader](https://docs.pytorch.org/docs/stable/data.html). Training/eval dataset. This wraps around a [torch.utils.data.Dataset](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.Dataset).
    - In short (somewhat simplified): a Dataset returns dataset rows indexed by integer index. A Dataloader wraps around the Dataset to provide: batching, shuffling (optional), and preprocessing to transform the raw Dataset rows to pytorch-friendly batched `torch.Tensor`'s (aka "tensor-ize" the inputs).
- torch.nn.Module. Define your model architecture, eg using torch.nn.Linear, torch.nn.LayerNorm, etc. Also specify your loss function, like torch.nn.CrossEntropyLoss.
- torch.optim.Optimizer. Which optimization algorithm to use, like torch.optim.SGD or torch.optim.Adam.
- The train/test loop. A function that trains your model on a training dataset, and evaluates on a test set.

## Fashion-MNIST Dataset

We'll use the `torchvision` library's convenience functions to download+load the Fashion-MNIST dataset:

In [ ]:
# This will download the data to the current working directory if the data doesn't already exist
training_data_fashion_mnist = torchvision.datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=torchvision.transforms.ToTensor(),
)

# Download test data from open datasets.
test_data_fashion_mnist = torchvision.datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=torchvision.transforms.ToTensor(),
)

### Visualize Fashion-MNIST Dataset

For fun, let's visualize 5 examples of each class from the Fashion-MNIST dataset.
As you can see, these are grayscale, low-resolution (28x28 pixels) images of clothing images:

In [ ]:
visualize_image_classification_dataset(dataset=test_data_fashion_mnist, figsize=(10, 20))
print("Visualized Fashion-MNIST dataset examples!")

## Q1.a: DataLoaders

**Task**: given the above FashionMNIST datasets, instantiate the `train_dataloader_fashion_mnist` and `test_dataloader_fashion_mnist` with the following properties:

- `train_dataloader_fashion_mnist`: use a batchsize of `batch_size_fashion_mnist` (64), with no random shuffling.
- `test_dataloader_fashion_mnist`: use a batchsize of `batch_size_fashion_mnist` (64), with no random shuffling.

Also, populate these variables: `num_classes_fashion_mnist, image_shape_fashion_mnist`.

Tip: see the pytorch docs on [torch.utils.data.DataLoader](https://docs.pytorch.org/docs/stable/data.html) to learn how to use it.
The Fashion-MNIST train/test Datasets are "Map-style" datasets.
For this assignment, you won't need to define custom `collate_fn`, nor a custom `sampler`, nor do you need to deal with multi-process data loading (ie you can keep `num_workers=0`, the default value).

Hint: once you've constructed your dataloader, you can view an example batch with this expression:

```
# X.shape=[B, C, H, W], dtype=torch.float32
# y.shape=[B], dtype=torch.int64
X, y = next(iter(train_dataloader_fashion_mnist))
```

In [ ]:
batch_size_fashion_mnist = 64

# Create data loaders.
# BEGIN YOUR CODE
train_dataloader_fashion_mnist = None
test_dataloader_fashion_mnist = None

# Number of classes in the FashionMNIST dataset
num_classes_fashion_mnist: int = None

# [channels, height_px, width_px]
image_shape_fashion_mnist: tuple[int, int, int] = None
# END YOUR CODE

### Populate submission dict

In [ ]:
# IMPORTANT: DO NOT MODIFY THIS CELL
# Run this cell to populate your submission dict
student_data["output"]["q1a_num_classes_fashion_mnist"] = num_classes_fashion_mnist
student_data["output"]["q1a_image_shape_fashion_mnist"] = image_shape_fashion_mnist

Once you've completed the above cell correctly, the following cell should produce a nice visualization of your dataloader by visualizing the first 5 test dataset batches:

In [ ]:
visualize_image_classification_dataloader(dataloader=test_dataloader_fashion_mnist, figsize=(10, 20))
print("Visualized Fashion-MNIST dataloader examples!")

## Q1.b SimpleMLPImageClassifier

Next, we'll define a simple image classifier model architecture:

<p align="center">
  <img src="https://github.com/data-188-berkeley/hw3/blob/main/figures/hw3_simple_image_classifier_model_arch.png?raw=1" alt="Model architecture for SimpleMLPImageClassifier"/>
</p>

**Task**: fill in the `SimpleMLPImageClassifier` class definition to achieve the desired model architecture and behavior.

Hints:
- For flattening the input image from `[B, C, H, W]` to `[B, C*H*W]`, you have a few options (feel free to choose whichever you prefer, they all can work):
  - [torch.Tensor::flatten()](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.flatten.html).
    - Tip: be sure to set `start_dim=1`, otherwise this will return `[B*C*H*W]`
  - [torch.flatten()](https://docs.pytorch.org/docs/stable/generated/torch.flatten.html#torch.flatten)
    - Tip: be sure to set `start_dim=1`, otherwise this will return `[B*C*H*W]`
  - [torch.nn.Flatten()](https://docs.pytorch.org/docs/stable/generated/torch.nn.modules.flatten.Flatten.html)
    - This is a Module that you'd add to your model definition. Interestingly, unlike the functional forms, for this layer `start_dim` defaults to `1`, which is what we want.

In [ ]:
class SimpleMLPImageClassifier(torch.nn.Module):
    """A simple image MLP classifier, with the following model architecture:
        x_img -> flatten -> linear -> relu -> linear -> relu -> linear -> logits
    """
    def __init__(self, image_shape: tuple[int, int, int], num_classes: int, hidden_dim: int = 512):
        """
        Args:
            image_shape: (C, H, W). Expected shape of input image.
            num_classes: int. Number of target classes in classification dataset.
            hidden_dim: int. Size ("width") of intermediate Linear layers.
        """
        super().__init__()
        self.image_shape = image_shape
        self.num_classes = num_classes
        self.hidden_dim = hidden_dim

        # BEGIN YOUR CODE
        # Hint: create the layers you need here (eg torch.nn.Linear, torch.nn.Relu).

        # END YOUR CODE

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Given input image, generate predicted logits.
        Args:
            x: torch.Tensor. Input image. shape=[B, C, H, W]
        Returns:
            pred_logits: torch.Tensor. shape=[B, num_classes]
        """
        # BEGIN YOUR CODE
        # Calculate and return predicted logits
        # Hint: flatten the input image here first
        pred_logits = None
        # END YOUR CODE

        return pred_logits

When you've completed the above cell, the following cell should run correctly:

In [ ]:
# Create model
model_q1b = SimpleMLPImageClassifier(image_shape=image_shape_fashion_mnist, num_classes=num_classes_fashion_mnist, hidden_dim=512)
print(model_q1b)
num_model_params_q1b = count_model_parameters(model_q1b)
print(f"Num model parameters:", num_model_params_q1b)

if num_model_params_q1b != 669706:
    # If you're failing this check, double check that you exactly implemented
    # the model architecture outlined in the spec, and that you're instantiating
    # the model with hidden_dim=512
    raise ValueError(f"Expected 669706 parameters, found {num_model_params_q1b}")

# Run forward pass on mock input image
# shape=[B, C, H, W]
mock_image_batch = torch.rand(size=(4, *image_shape_fashion_mnist), dtype=torch.float32)
mock_out_q1b = model_q1b(mock_image_batch)
print(f"Mock output shape:", mock_out_q1b.shape)

if mock_out_q1b.shape != (4, num_classes_fashion_mnist):
    raise ValueError(f"Expected mock output shape to be {(4, num_classes_fashion_mnist)}, found {mock_out_q1b.shape}")


### Populate submission dict

In [ ]:
# IMPORTANT: DO NOT MODIFY THIS CELL
# Run this cell to populate your submission dict
student_data["output"]["q1b_num_model_params_q1b"] = num_model_params_q1b
student_data["output"]["q1b_mock_out_shape"] = mock_out_q1b.shape

## Q1.c: Train loop

Next, we'll implement the train loop!

**Task:** Implement the `train_epoch()` function that performs a single epoch of training.

`train_epoch()` should return a dict `out_meta` with the following keys:

- `"loss"`: a `list[float]` containing the loss value for each logged batch (output volume controlled by `log_every_n_steps`).
    - Tip: use `Tensor:item()` to convert the scalar loss value `Tensor` to a `float`. If you don't do this, memory usage may grow too high (ex: due to undesired computation graph growth)!
- `"ind_batch"`: A `list[int]` containing the batch index corresponding to the values in `"loss"`. Use zero-based indexing, ie this should always start with `0`.
- `"dur_total_secs"`: the total amount of time to train this epoch (in seconds).
- `"tput_total"`: the total training throughput, in samples/sec.
    - Tip: use `len(dataloader)` to get the total number of dataloader batches, and `len(dataloader.dataset)` to get the total number of dataset samples.

Some additional behavior of note:

- `log_every_n_steps`: Every n batches, print the current loss to stdout, and:
    - Append the current batch loss value to `out_meta["loss"]`
        - Also append the current batch index (0-based indexing) to: `out_meta["ind_batch"]`
    - The first batch should always be logged to `out_meta`.
    - If the number of batches is not evenly divisible by `log_every_n_steps`, then ensure that the final batch is logged to `out_meta`.
    - Ex: if `num_train_batches=10`, and `log_every_n_steps=6`, then there should be 3 entries in `out_meta`:
        - `batch=0:` log batch=0 (first batch, always)
        - `batch=1:` don't log
        - ...
        - `batch=6:` log batch=6 (because `log_every_n_steps=6`)
        - `batch=7:` don't log
        - ...
        - `batch=9:` log batch=9 (final batch)

Hints:
- Use `model.train()` to ensure that the model is in "training" mode (ex: important for layers like BatchNorm)
- Use this pattern to implement the behavior of "every n batches, do something, and ensure last batch is logged":

```
for ind_batch, (X, y) in enumerate(dataloader):
    ...
    if (ind_batch % log_every_n_steps == 0) or (ind_batch == (len(dataloader) - 1)):
        # log data
```
- For FashionMNIST, with our default provided batchsize (64), there are 938 train batches (`len(train_dataloader_fashion_mnist)`). Thus, your final `ind_batch` in your `out_meta_train` should be: `937`.


Hint: you may find the `time` built-in module helpful for measuring execution time:

In [ ]:
import time

def some_function(sleep_dur_secs: float):
    print(f"Sleeping for {sleep_dur_secs} secs...")
    time.sleep(sleep_dur_secs)
    print(f"Woke up!")

# Measure the execution time of the following code block
tic_start = time.time()

some_function(sleep_dur_secs=3.2) # say this takes 3.2 seconds

# duration_secs will be 3.2
duration_secs = time.time() - tic_start
print(f"duration_secs: {duration_secs}")

In [ ]:
# Train loop
def train_epoch(
        dataloader: torch.utils.data.DataLoader,
        model: torch.nn.Module,
        loss_fn: torch.nn.Module,
        optimizer: torch.optim.Optimizer,
        log_every_n_steps: int = 100
    ) -> dict[str, Any]:
    """Trains the model on an input training dataloader.
    Args:
        dataloader:
        model:
        loss_fn:
        optimizer:
        log_every_n_steps: Every n batches, print the current loss to stdout (and
            any other useful information), add any relevant metadata to
            `out_meta` (ex: the current batch's loss). See problem spec for
            more details.
    Returns:
        out_meta: dict[str, Any]. Useful training metadata. See problem spec
            to learn exactly what this should contain.
    """
    out_meta = {}
    # BEGIN YOUR SOLUTION
    # END YOUR SOLUTION

    return out_meta

## Q1.d Test loop

Implement the `test()` function that calculates offline classification evaluation metrics on a test set, namely:
- **Accuracy**. Calculate the accuracy over the test dataset, defined as:
    - `accuracy = num_preds_correct / num_test_samples`

`test()` should output a dict `out_meta` with the following keys:
- `"test_loss"`: total test loss, averaged by num test batches.
- `"test_accuracy"`: test accuracy.
- `"dur_total_secs"`: total time to run evaluation, in units seconds.
- `"tput_total"`: total evaluation throughput, in units samples/sec.

In [ ]:
# Evaluation
def test(
        dataloader: torch.utils.data.DataLoader,
        model: torch.nn.Module,
        loss_fn: torch.nn.Module
    ) -> dict[str, Any]:
    """Evaluates a classification model on a test dataset.
    Args:
        dataloader: Test dataset.
        model:
        loss_fn: Loss function to use, eg CrossEntropyLoss.
    Returns:
        out_meta: dict[str, Any]. Useful evaluation metadata. See problem spec
            to learn exactly what this should contain.
    """
    out_meta = {}
    # BEGIN YOUR SOLUTION
    # END YOUR SOLUTION

    return out_meta

## Q1.e Train your model!

Once you've implemented the train and test loops, run the below cells to train and evaluate your model!

Note that we've done the following:
- Assign `loss_fn` to use CrossEntropyLoss.
- Assign `optimizer` to use an SGD optimizer with the following:
    - learning rate of `1e-3`
    - momentum enabled, with a momentum factor of `0.9`
    - L2 regularization disabled

You should be able to achieve ~80% test accuracy with a train throughput of ~3000 - 5300 samples/sec.
It should take ~60 - 80 seconds to run the below training+eval cell.

**To avoid issues with the autograder:** please leave all parameters unchanged, eg: `batch_size_fashion_mnist=64, num_epochs_fashion_mnist=5, log_every_n_steps_fashion_mnist=100`.
To pass the autograder, you must achieve a test accuracy of at least 0.75 (inclusive. achieved at the end of any train epoch).

In [ ]:
# Create model, train+eval
model_q1e = SimpleMLPImageClassifier(image_shape=image_shape_fashion_mnist, num_classes=num_classes_fashion_mnist, hidden_dim=512)
print(model_q1e)
num_params_q1e = count_model_parameters(model_q1e)
print(f"Num model parameters:", num_params_q1e)

loss_fn_q1e = torch.nn.CrossEntropyLoss()
optimizer_q1e = torch.optim.SGD(model_q1e.parameters(), lr=1e-3, momentum=0.9)

num_epochs_fashion_mnist = 5
log_every_n_steps_fashion_mnist = 100
out_metas_train_q1e, out_metas_test_q1e = train_epochs(
    model=model_q1e,
    train_dataloader=train_dataloader_fashion_mnist,
    train_epoch_fn=train_epoch,
    test_dataloader=test_dataloader_fashion_mnist,
    test_fn=test,
    loss_fn=loss_fn_q1e,
    optimizer=optimizer_q1e,
    num_epochs=num_epochs_fashion_mnist,
    log_every_n_steps=log_every_n_steps_fashion_mnist,
)
print("Done!")

In [ ]:
# Perform a few checks. You must pass this cell to pass the autograder.
check_metas_train_test(
    out_metas_train=out_metas_train_q1e,
    out_metas_test=out_metas_test_q1e,
    num_train_batches=len(train_dataloader_fashion_mnist),
    log_every_n_steps=log_every_n_steps_fashion_mnist,
    num_train_epochs=num_epochs_fashion_mnist,
)
max_test_acc = max([out_meta_test["test_accuracy"] for out_meta_test in out_metas_test_q1e])
if max_test_acc < 0.75:
    raise RuntimeError(f"Max test acc achieved was {max_test_acc}, but did not exceed min required 0.75.")
print("Passed!")

### Populate submission dict

In [ ]:
# IMPORTANT: DO NOT MODIFY THIS CELL
# Run this cell to populate your submission dict
student_data["output"]["q1e_out_metas_train_q1e"] = out_metas_train_q1e
student_data["output"]["q1e_out_metas_test_q1e"] = out_metas_test_q1e

## Save Model

Let's save your trained model to disk.

In [ ]:
# Save model snapshot
outpath_q1_fashion_mnist_snapshot = "model_q1_fashion_mnist.pth"
torch.save(model_q1e.state_dict(), outpath_q1_fashion_mnist_snapshot)
print(f"Saved PyTorch Model State to: {outpath_q1_fashion_mnist_snapshot}")

In [ ]:
# (check) load model, evaluate on test set again
model_from_disk = SimpleMLPImageClassifier(image_shape=image_shape_fashion_mnist, num_classes=num_classes_fashion_mnist)
model_from_disk.load_state_dict(torch.load(outpath_q1_fashion_mnist_snapshot, weights_only=True))
test(test_dataloader_fashion_mnist, model_from_disk, loss_fn_q1e)

## Visualize predictions

Finally, let's visualize the predictions of your trained model!

In [ ]:
visualize_image_classifier_preds(
    model=model_q1e,
    dataloader=test_dataloader_fashion_mnist,
    class_names=test_data_fashion_mnist.classes,
    image_shape=image_shape_fashion_mnist,
    top_k=5,
)

# Question 2: CIFAR10 Classification [15 pts]

In this section, we'll see that it's easy to apply our above pipeline to a different dataset.
We'll explore the CIFAR-10 image classification dataset, which is generally considered to be more challenging than MNIST and Fashion-MNIST.

From the [website](https://www.cs.toronto.edu/~kriz/cifar.html): "The CIFAR-10 dataset consists of 60000 32x32 colour images in 10 classes, with 6000 images per class. There are 50000 training images and 10000 test images."

<p align="center">
  <img src="https://github.com/data-188-berkeley/hw3/blob/main/figures/cifar10_examples.png?raw=1" alt="CIFAR10 dataset examples"/>
</p>




In [ ]:
# Download training data
# This will download the data to the current working directory if the data doesn't already exist
training_data_cifar10 = torchvision.datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=torchvision.transforms.ToTensor(),
)

# Download test data from open datasets.
test_data_cifar10 = torchvision.datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=torchvision.transforms.ToTensor(),
)

## Visualize CIFAR-10 Dataset

Let's visualize the CIFAR-10 Dataset:

In [ ]:
visualize_image_classification_dataset(dataset=test_data_cifar10, figsize=(10, 20))
print("Visualized CIFAR-10 dataset examples!")

## Q2.a Dataloaders

**Task**: given the CIFAR10 datasets, instantiate the `train_dataloader_cifar10` and `test_dataloader_cifar10` with the following properties:

- `train_dataloader_cifar10`: use a batchsize of `batch_size_cifar10` (64), with no random shuffling.
- `test_dataloader_cifar10`: use a batchsize of `batch_size_cifar10` (64), with no random shuffling.

Also, populate these variables: `num_classes_cifar10, image_shape_cifar10`.

Hint: once you've constructed your dataloader, you can view an example batch with this expression:

```
# X.shape=[B, C, H, W], dtype=torch.float32
# y.shape=[B], dtype=torch.int64
X, y = next(iter(train_dataloader_cifar10))
```

In [ ]:
batch_size_cifar10 = 64
# Create data loaders.

# BEGIN YOUR CODE
train_dataloader_cifar10 = None
test_dataloader_cifar10 = None

# Number of classes in the CIFAR-10 dataset
num_classes_cifar10: int = None

# [channels, height_px, width_px]
image_shape_cifar10: tuple[int, int, int] = None
# END YOUR CODE

### Populate submission dict

In [ ]:
# IMPORTANT: DO NOT MODIFY THIS CELL
# Run this cell to populate your submission dict
student_data["output"]["q2a_num_classes_cifar10"] = num_classes_cifar10
student_data["output"]["q2a_image_shape_cifar10"] = image_shape_cifar10

Once you've completed the dataloaders, the below cell should produce a nice visualization of the first 5 batches of your test dataloader:

In [ ]:
visualize_image_classification_dataloader(dataloader=test_dataloader_cifar10, figsize=(10, 20))
print("Visualized CIFAR-10 dataloader examples!")

## Q2.b Train your model!

**Task:** Complete the cell to instantiate a `SimpleMLPImageClassifier` (hidden_dim=512) appropriate for CIFAR-10, and train+evaluate your model!

You should be able to achieve ~40% test accuracy, with a train throughput of ~1800 examples/sec.
It should take you about ~140 seconds to run the below cell.

**To avoid issues with the autograder:** please leave all parameters unchanged, eg: `batch_size_cifar10=64, num_epochs_cifar10=5, log_every_n_steps_cifar10=100`, etc.
To pass the autograder, you must achieve a test accuracy of at least 0.35 (inclusive. achieved at the end of any train epoch).

In [ ]:
# Create model, train+test
# BEGIN YOUR SOLUTION
model_cifar10_q2b = None
# END YOUR SOLUTION

print(model_cifar10_q2b)
num_model_params_q2b = count_model_parameters(model_cifar10_q2b)
print(f"Num model parameters: {num_model_params_q2b}")

# Train, eval
loss_fn_q2b = torch.nn.CrossEntropyLoss()
optimizer_q2b = torch.optim.SGD(model_cifar10_q2b.parameters(), lr=1e-3, momentum=0.9)

num_epochs_cifar10 = 5
log_every_n_steps_cifar10 = 100
out_metas_train_q2b, out_metas_test_q2b = train_epochs(
    model=model_cifar10_q2b,
    train_dataloader=train_dataloader_cifar10,
    train_epoch_fn=train_epoch,
    test_dataloader=test_dataloader_cifar10,
    test_fn=test,
    loss_fn=loss_fn_q2b,
    optimizer=optimizer_q2b,
    num_epochs=num_epochs_cifar10,
    log_every_n_steps=log_every_n_steps_cifar10,
)
print("Done!")

In [ ]:
# Perform a few checks. You must pass this cell to pass the autograder.
check_metas_train_test(
    out_metas_train=out_metas_train_q2b,
    out_metas_test=out_metas_test_q2b,
    num_train_batches=len(train_dataloader_cifar10),
    log_every_n_steps=log_every_n_steps_cifar10,
    num_train_epochs=num_epochs_cifar10,
)
max_test_acc = max([out_meta_test["test_accuracy"] for out_meta_test in out_metas_test_q2b])
if max_test_acc < 0.35:
    raise RuntimeError(f"Max test acc achieved was {max_test_acc}, but did not exceed min required 0.35.")
print("Passed!")

### Populate submission dict

In [ ]:
# IMPORTANT: DO NOT MODIFY THIS CELL
# Run this cell to populate your submission dict
student_data["output"]["q2b_out_metas_train_q2b"] = out_metas_train_q2b
student_data["output"]["q2b_out_metas_test_q2b"] = out_metas_test_q2b

## Visualize predictions

Finally, let's visualize the predictions of your trained model!

In [ ]:
visualize_image_classifier_preds(
    model=model_cifar10_q2b,
    dataloader=test_dataloader_cifar10,
    class_names=test_data_cifar10.classes,
    image_shape=image_shape_cifar10,
    top_k=5,
)

# Save progress, and next notebook!

**Important:** to correctly capture your submission for the autograder, you must have run all cells in this notebook's **current session** before running the below cell

If your autograder submission fails due to missing data, please re-run this entire notebook.

Please run the following cell to save your outputs for the Gradescope autograder:

In [ ]:
# Save the student_data to disk
save_student_data(student_data=student_data, outpath_student_data=OUTPATH_STUDENT_DATA_CPU)

Next, move onto the next notebook, hw3_gpu.ipynb!